In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Extraction du "Vocabolario toscano dell'arte del disegno" de Filippo
Baldinucci depuis :
    https://baldinucci.accademiadellacrusca.org/testo-del-vocabolario
vers un fichier XML de type TEI, structuré ainsi :

    <vocabolario fonte="...">
      <div1 type="letter" n="A">
        <entry xml:id="abaco">
          <form>Abaco</form>
          <def>...</def>
        </entry>
        ...
      </div1>
      ...
    </vocabolario>

La page conserve, pour chaque entrée du dictionnaire, une balise <form>
(la vedette / le lemme) suivie d'une ou plusieurs balises <p> (la ou les
définitions). Le sommaire en haut de page contient des ancres du type
<a href="#A">, <a href="#B">, ... qui pointent vers des éléments portant
ces mêmes id dans le corps du texte : c'est ce qu'on utilise ici pour
découper le document par lettre.

Installation :
    pip install requests beautifulsoup4 lxml

Utilisation :
    python baldinucci_to_xml.py
"""

import re
import sys
import unicodedata
from xml.etree import ElementTree as ET
from xml.dom import minidom

import requests
from bs4 import BeautifulSoup, Tag

URL = "https://baldinucci.accademiadellacrusca.org/testo-del-vocabolario"
OUTPUT_FILE = "vocabolario_baldinucci_tei.xml"
DEBUG_HTML_FILE = "debug_page.html"

# Lettres du sommaire de la page (ancres #A, #B, ... #UV, #Z, #AGG)
LETTERS = [
    "A", "B", "C", "D", "E", "F", "G", "H", "I", "L", "M", "N", "O", "P",
    "Q", "R", "S", "T", "UV", "Z"
]


def fetch_html(url: str) -> str:
    headers = {"User-Agent": "Mozilla/5.0 (compatible; VocabExtractor/1.0)"}
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding
    return resp.text


def clean_text(txt: str) -> str:
    txt = txt.replace("\xa0", " ")
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt


def slugify(text: str) -> str:
    """
    Transforme un lemme (ex: "Cerchiare.", "à ") en un xml:id valide :
    minuscules, sans accents, sans ponctuation ni espaces.
    """
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "-", text)
    text = text.strip("-")
    if not text:
        text = "x"
    # xml:id ne doit pas commencer par un chiffre
    if text[0].isdigit():
        text = "n" + text
    return text


def locate_letter_anchor(soup: BeautifulSoup, letter: str):
    """
    Cherche l'élément qui sert de point d'ancrage à une lettre dans le
    corps du texte (pas dans le sommaire A-Z en haut de page).
    """
    candidates = soup.find_all(id=letter)
    if not candidates:
        candidates = soup.find_all(attrs={"name": letter})
    if not candidates:
        return None
    if len(candidates) == 1:
        return candidates[0]
    for el in candidates:
        if el.name != "a":
            return el
    return candidates[-1]


def build_xml(soup: BeautifulSoup) -> ET.Element:
    root = ET.Element("vocabolario", attrib={"fonte": URL})

    body = soup.body or soup

    anchor_map = {}
    for letter in LETTERS:
        el = locate_letter_anchor(soup, letter)
        if el is not None:
            anchor_map[id(el)] = letter

    if not anchor_map:
        raise RuntimeError(
            "Aucune ancre de lettre (id=A, id=B, ...) n'a été trouvée. "
            "La structure HTML réelle diffère de ce qui est supposé ici : "
            "ouvrez debug_page.html et ajustez locate_letter_anchor()."
        )

    used_ids = {}  # slug -> compteur, pour garantir l'unicité des xml:id

    current_div1 = None
    current_entry = None
    current_def_parts = []

    def flush_entry():
        """Ecrit les <p> accumulés dans le <def> de l'entrée en cours."""
        nonlocal current_entry, current_def_parts
        if current_entry is not None and current_def_parts:
            def_el = ET.SubElement(current_entry, "def")
            def_el.text = " ".join(current_def_parts)
        current_def_parts = []

    for el in body.find_all(True):
        if id(el) in anchor_map:
            flush_entry()
            letter = anchor_map[id(el)]
            current_div1 = ET.SubElement(
                root, "div1", attrib={"type": "letter", "n": letter}
            )
            current_entry = None
            continue

        if current_div1 is None:
            continue  # avant la première lettre repérée : on ignore

        if el.name == "form":
            flush_entry()
            forma_txt = clean_text(el.get_text())
            if not forma_txt:
                continue

            slug = slugify(forma_txt)
            if slug in used_ids:
                used_ids[slug] += 1
                xml_id = f"{slug}-{used_ids[slug]}"
            else:
                used_ids[slug] = 1
                xml_id = slug

            current_entry = ET.SubElement(
                current_div1, "entry",
                attrib={"{http://www.w3.org/XML/1998/namespace}id": xml_id},
            )
            forma_el = ET.SubElement(current_entry, "form")
            forma_el.text = forma_txt
            continue

        if el.name == "p":
            txt = clean_text(el.get_text())
            if not txt:
                continue
            if current_entry is None:
                # <p> rencontré avant tout <form> dans cette lettre :
                # on crée une entrée "orpheline" pour ne pas perdre le texte
                slug = "sans-forme"
                if slug in used_ids:
                    used_ids[slug] += 1
                    xml_id = f"{slug}-{used_ids[slug]}"
                else:
                    used_ids[slug] = 1
                    xml_id = slug
                current_entry = ET.SubElement(
                    current_div1, "entry",
                    attrib={"{http://www.w3.org/XML/1998/namespace}id": xml_id},
                )
            current_def_parts.append(txt)

    flush_entry()  # ne pas oublier la toute dernière entrée

    return root


def prettify(elem: ET.Element) -> str:
    ET.register_namespace("xml", "http://www.w3.org/XML/1998/namespace")
    rough = ET.tostring(elem, encoding="utf-8")
    return minidom.parseString(rough).toprettyxml(indent="  ", encoding="utf-8").decode("utf-8")


def main():
    print(f"Téléchargement de {URL} ...")
    try:
        html = fetch_html(URL)
    except requests.RequestException as e:
        print(f"Erreur réseau : {e}", file=sys.stderr)
        sys.exit(1)

    print("Analyse du HTML ...")
    soup = BeautifulSoup(html, "lxml")

    print("Construction du XML ...")
    try:
        root = build_xml(soup)
    except RuntimeError as e:
        print(f"Erreur : {e}", file=sys.stderr)
        with open(DEBUG_HTML_FILE, "w", encoding="utf-8") as f:
            f.write(html)
        print(f"HTML brut sauvegardé dans {DEBUG_HTML_FILE} pour inspection.")
        sys.exit(1)

    nb_div1 = len(root.findall("div1"))
    nb_entry = len(root.findall(".//entry"))
    print(f"{nb_div1} sections de lettres, {nb_entry} entrées trouvées.")

    if nb_entry == 0:
        print(
            "ATTENTION : aucune entrée trouvée à l'intérieur des sections "
            "de lettres. La structure HTML réelle diffère probablement de "
            "ce qui est supposé dans ce script. Le HTML brut va être "
            "sauvegardé pour inspection.",
            file=sys.stderr,
        )
        with open(DEBUG_HTML_FILE, "w", encoding="utf-8") as f:
            f.write(html)
        print(f"HTML brut sauvegardé dans {DEBUG_HTML_FILE}.")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(prettify(root))

    print(f"Fichier XML écrit : {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Téléchargement de https://baldinucci.accademiadellacrusca.org/testo-del-vocabolario ...
Analyse du HTML ...
Construction du XML ...
20 sections de lettres, 4043 entrées trouvées.
Fichier XML écrit : vocabolario_baldinucci_tei.xml
